<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/0f/We_logo.svg/3840px-We_logo.svg.png" alt="WE Logo" width="60" style="margin-bottom:8px"/>

# Telecom Egypt (WE) — ML Project
## Customer Complaint Classification, Summarization & Sentiment Analysis

---

### Background

Telecom Egypt (WE) receives thousands of customer support tickets every day across multiple channels — call center transcripts, the WE app chat, and social media comments. Today, agents read every ticket in full before they can tag its category, gauge how upset the customer is, and decide who should handle it first. This is slow, inconsistent between agents, and makes it hard to spot urgent cases early.

This project simulates a real-world NLP task at WE. You are part of the Data & AI team, and — unlike the Smart Plan project — **there is no historical ticket dataset waiting for you.** The Customer Care team has never systematically logged ticket text before, so before you can train anything, you first have to **construct a realistic synthetic dataset** yourselves, then use it to build an NLP pipeline that, given the raw text of a customer ticket, can:

> **Classify** — which department/category should own this ticket (Billing, Network, Technical Support, Sales/Plans, Other)
> **Summarize** — reduce a long, messy complaint into a 1–2 sentence summary an agent can read in seconds
> **Detect Sentiment** — flag whether the customer is Positive, Neutral, or Negative, so angry customers can be prioritized



---

### General Instructions

- Read each step carefully before writing any code.
- Each step has a clear objective — understand **why** you are doing it, not just **how**.
- Ticket text should be messy on purpose: Arabic, English, Franco-Arabic (Arabic written in Latin letters), emojis, and typos. A too-clean synthetic dataset will teach you nothing about real preprocessing challenges.
- You are expected to make decisions along the way (e.g., how many categories, how imbalanced, how to simulate missing ratings). Document your reasoning in markdown cells.
- There is no single correct answer — justify your choices.


### 1. Environment Setup
Installs necessary NLP libraries (`transformers`, `langdetect`) and checks for GPU availability.

In [1]:
# ── Block 1: Installs & Imports ──────────────────────────────────────────
!pip install -q transformers sentencepiece accelerate langdetect

import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Using GPU


### 2. Sample Data
Defines a small list of multi-lingual (Arabic, English, Mixed) complaints to test the initial logic.

In [2]:
# ── Block 2: Sample Complaints (EN / AR / Mixed) ─────────────────────────
sample_complaints = [
    "I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!",
    "الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت",
    "My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.",
    "خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة",
    "The new mobile plan is a total ripoff, I'm paying more for less data than last month."
]

for i, c in enumerate(sample_complaints):
    print(f"[{i}] {c}\n")

[0] I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!

[1] الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت

[2] My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.

[3] خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة

[4] The new mobile plan is a total ripoff, I'm paying more for less data than last month.



### 3. Language Routing
Functions to split text into sentences and detect the language of each part to route them to the correct model.

In [3]:
# ── Block 3: Language Detection & Sentence-Level Splitting ──────────────
import re
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def split_sentences(text):
    # Split on Arabic/English sentence enders and newlines, keep it simple
    parts = re.split(r'(?<=[.!?؟\n])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def detect_lang_chunks(text):
    """Split a complaint into (lang, chunk) pairs for Sys 2 routing."""
    chunks = split_sentences(text)
    result = []
    for chunk in chunks:
        try:
            lang = detect(chunk)
        except:
            lang = "unknown"
        lang_bucket = "ar" if lang == "ar" else "en"
        result.append((lang_bucket, chunk))
    return result

# Test on our samples
for i, c in enumerate(sample_complaints):
    print(f"[{i}] ORIGINAL: {c}")
    print(f"     SPLIT: {detect_lang_chunks(c)}\n")

[0] ORIGINAL: I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!
     SPLIT: [('en', "I've been charged twice this month for my internet bill and no one is responding to my emails."), ('en', 'This is so frustrating!')]

[1] ORIGINAL: الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت
     SPLIT: [('ar', 'الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت')]

[2] ORIGINAL: My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.
     SPLIT: [('ar', 'My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل.'), ('en', 'I need this fixed today.')]

[3] ORIGINAL: خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة
     SPLIT: [('ar', 'خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة')]

[4] ORIGINAL: The new mobile p

In [4]:
# ── Block 3b: is_arabic helper (moved earlier — needed by Sys1 summarizer) ─
def is_arabic(text):
    arabic_chars = sum(1 for ch in text if '\u0600' <= ch <= '\u06FF')
    return arabic_chars > len(text) * 0.3

In [5]:
# ── Block 4: Sys 1 — Multilingual BERT (Sentiment + Embedding Classification) ─
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

# Sentiment: BERT fine-tuned on multilingual review sentiment (AR+EN)
sys1_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device
)

# Embedding model: plain multilingual BERT
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
mbert_model = AutoModel.from_pretrained("bert-base-multilingual-cased").to(
    "cuda" if device == 0 else "cpu"
).eval()

def bert_embed(text):
    dev = "cuda" if device == 0 else "cpu"
    inputs = mbert_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(dev)
    with torch.no_grad():
        outputs = mbert_model(**inputs)
    # mean-pool token embeddings
    emb = outputs.last_hidden_state.mean(dim=1)
    return emb

CATEGORY_LABELS = ["Billing", "Network", "Technical Support", "Sales/Plans", "Other"]
category_embeds = {label: bert_embed(label) for label in CATEGORY_LABELS}

def classify_by_embedding(text):
    text_emb = bert_embed(text)
    scores = {label: F.cosine_similarity(text_emb, emb).item() for label, emb in category_embeds.items()}
    best = max(scores, key=scores.get)
    return best, scores

# Quick test
for i, c in enumerate(sample_complaints):
    sent = sys1_sentiment(c[:512])[0]
    cat, scores = classify_by_embedding(c)
    scores_rounded = {k: round(v, 2) for k, v in scores.items()}
    print(f"[{i}] TEXT: {c}")
    print(f"     SENTIMENT: {sent}")
    print(f"     CATEGORY: {cat} | scores: {scores_rounded}\n")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0] TEXT: I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!
     SENTIMENT: {'label': '1 star', 'score': 0.9032513499259949}
     CATEGORY: Technical Support | scores: {'Billing': 0.22, 'Network': 0.23, 'Technical Support': 0.27, 'Sales/Plans': 0.26, 'Other': 0.26}

[1] TEXT: الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت
     SENTIMENT: {'label': '1 star', 'score': 0.49976158142089844}
     CATEGORY: Sales/Plans | scores: {'Billing': 0.28, 'Network': 0.21, 'Technical Support': 0.27, 'Sales/Plans': 0.31, 'Other': 0.22}

[2] TEXT: My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.
     SENTIMENT: {'label': '1 star', 'score': 0.40460771322250366}
     CATEGORY: Technical Support | scores: {'Billing': 0.26, 'Network': 0.29, 'Technical Support': 0.35, 'Sales/Plans': 0.27, 'Other': 0.24}

[3] TEXT: خدمة م

### 4. System 1: Baseline Models
Initializes multi-lingual BERT for sentiment analysis and zero-shot category classification using embeddings.

In [6]:
# ── Block 5 (Revised): Expanded Synthetic Labeled Dataset ───────────────
import pandas as pd
import random

random.seed(42)

category_templates = {
    "Billing": [
        "I've been charged twice for my {service} this month, please refund me.",
        "اتخصم مني مبلغ غلط في فاتورة {service} الشهر ده",
        "My invoice shows a charge I don't recognize for {service}.",
        "الفاتورة بتاعتي غلط للمرة كذا، حد يراجعها لو سمحتوا",
        "Why was I billed more than my usual {service} plan price?",
        "دفعت المبلغ واتخصم مرتين، محتاج استرداد فلوسي",
        "There's an extra fee on my {service} bill that I never agreed to.",
        "مش فاهم ليه الفاتورة زادت من غير سبب واضح",
        "I paid my bill but the system still shows it as unpaid.",
        "طلبت استرداد فلوس من كذا يوم ومحدش رد",
        "Can you explain this unexpected charge on my account?",
        "الخصم من رصيدي مش مطابق للباقة اللي انا مشترك فيها",
        "I was double billed for the same {service} package.",
        "عايز افهم ليه اتحاسبت على خدمة انا مش مشترك فيها",
        "My payment went through but my balance wasn't updated.",
    ],
    "Network": [
        "My internet has been down for {n} days now with no fix.",
        "الشبكة عندي واقعة من {n} يوم ومفيش حد بيرد",
        "No signal at all in my area since yesterday.",
        "الانترنت بطيء جدا في منطقتي من كذا يوم",
        "Constant network drops during calls, very frustrating.",
        "مفيش تغطية خالص في البيت من كام يوم",
        "My {service} connection keeps disconnecting every few minutes.",
        "الاتصال بيقطع كل شوية وانا مش عارف اكمل شغلي",
        "There's been an outage in my neighborhood for hours.",
        "في عطل في الشبكة من الصبح ومفيش تحديث",
        "The signal strength is very weak inside my house.",
        "التغطية ضعيفة جدا في المنطقة دي من زمان",
        "I can't make calls because there's no network coverage.",
        "مفيش نت خالص من امبارح، الموضوع بقى متكرر",
        "Download speeds are extremely slow compared to what I'm paying for.",
    ],
    "Technical Support": [
        "My router keeps resetting itself, can someone help?",
        "الراوتر بتاعي مش شغال، جربت اعيد تشغيله ومفيش فايدة",
        "The app keeps crashing every time I try to log in.",
        "التطبيق بيقفل لوحده كل ما افتحه",
        "I need technical help setting up my new modem.",
        "محتاج دعم فني عشان اظبط الجهاز الجديد",
        "My device won't connect to the {service} network at all.",
        "مش عارف اظبط اعدادات الراوتر لوحدي محتاج مساعدة فنية",
        "The website keeps giving me an error when I try to log in.",
        "التطبيق مش بيفتح خالص من كذا يوم، جربت احدثه ومفيش فايدة",
        "Can a technician come check my line, it's not working properly?",
        "عايز فني يجي يكشف على الخط عندي في البيت",
        "I followed the setup guide but the device still isn't working.",
        "جربت كل الحلول المكتوبة بس المشكلة لسه موجودة",
        "My SIM card isn't being recognized by my phone anymore.",
    ],
    "Sales/Plans": [
        "I want to upgrade my {service} plan to unlimited data.",
        "عايز اغير الباقة بتاعتي لباقة اكبر",
        "What are the current offers on new mobile plans?",
        "في عروض جديدة على باقات النت؟",
        "The new plan is a ripoff, paying more for less data.",
        "الباقة الجديدة غالية جدا مقارنة بالقديمة",
        "Can I switch from prepaid to a postpaid {service} plan?",
        "عايز اعرف ازاي احول من الباقة المسبقة الدفع للباقة الشهرية",
        "I'd like to downgrade my plan, it's too expensive for me.",
        "الاسعار غالية اوي مقارنة بالشركات التانية",
        "Is there a family plan option available for {service}?",
        "فيه باقات عيلية ولا كل حد لوحده؟",
        "I want to cancel my current plan and switch to another one.",
        "عايز الغي الاشتراك واشترك في باقة تانية",
        "What's included in the premium {service} package?",
    ],
    "Other": [
        "How do I update my personal information on the account?",
        "عايز اغير رقم التليفون المسجل على حسابي",
        "Just wanted to say thank you for the quick support yesterday.",
        "شكرا على الخدمة الممتازة والسرعة في الحل",
        "Where is the nearest WE store to my location?",
        "عايز اعرف اقرب فرع لمنطقتي",
        "What are your customer service working hours?",
        "مواعيد خدمة العملاء ايه بالظبط؟",
        "I really appreciate how helpful the agent was on the call.",
        "الموظف كان محترم جدا وساعدني بسرعة، شكرا ليكم",
        "How can I close my account permanently?",
        "عايز اقفل حسابي نهائيا، ازاي اعمل كده؟",
        "Do you have a mobile app I can download?",
        "فيه تطبيق موبايل احمله عشان اتابع حسابي؟",
        "General question about how loyalty points work.",
    ],
}

services = ["internet", "mobile", "landline", "TV"]
n_per_category = 60

rows = []
for category, templates in category_templates.items():
    for _ in range(n_per_category):
        t = random.choice(templates)
        text = t.format(service=random.choice(services), n=random.randint(2, 7))
        rows.append({"text": text, "category": category})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(df.shape)
print(df["category"].value_counts())
df.head(10)

(300, 2)
category
Sales/Plans          60
Other                60
Technical Support    60
Billing              60
Network              60
Name: count, dtype: int64


,text,category
0,في عروض جديدة على باقات النت؟,Sales/Plans
1,How do I update my personal information on the...,Other
2,التطبيق بيقفل لوحده كل ما افتحه,Technical Support
3,I was double billed for the same mobile package.,Billing
4,I want to upgrade my internet plan to unlimite...,Sales/Plans
5,I want to upgrade my landline plan to unlimite...,Sales/Plans
6,عايز اعرف ازاي احول من الباقة المسبقة الدفع لل...,Sales/Plans
7,في عطل في الشبكة من الصبح ومفيش تحديث,Network
8,اتخصم مني مبلغ غلط في فاتورة mobile الشهر ده,Billing
9,عايز فني يجي يكشف على الخط عندي في البيت,Technical Support


### 5. Dataset Generation
Creates a synthetic labeled dataset of 300 samples across five categories (Billing, Network, etc.) using templates.

In [7]:
# ── Block 6: Train/Test Split & Tokenization ─────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])
label_names = list(le.classes_)
print("Label mapping:", dict(zip(le.transform(label_names), label_names)))

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# Reuse the mbert_tokenizer already loaded in Block 4
class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ComplaintDataset(train_df["text"], train_df["label"], mbert_tokenizer)
test_dataset = ComplaintDataset(test_df["text"], test_df["label"], mbert_tokenizer)

print("Sample item keys:", train_dataset[0].keys())

Label mapping: {np.int64(0): 'Billing', np.int64(1): 'Network', np.int64(2): 'Other', np.int64(3): 'Sales/Plans', np.int64(4): 'Technical Support'}
Train: 240 | Test: 60
Sample item keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


### 6. Preprocessing
Encodes labels and prepares the PyTorch Dataset for the classification model.

fine tuning

In [8]:
# ── Block 7: Fine-Tune BERT for Category Classification ─────────────────
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

num_labels = len(label_names)

category_model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=num_labels
).to("cuda" if device == 0 else "cpu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./category_model_ckpt",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    learning_rate=2e-5,
    report_to="none"
)

trainer = Trainer(
    model=category_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.603772,1.410555,0.750000,0.737776
2,1.217062,1.019365,0.783333,0.781628
3,0.876138,0.637862,0.933333,0.932659
4,0.432909,0.377573,0.933333,0.932659
5,0.255510,0.274761,0.933333,0.932659
6,0.118382,0.252974,0.933333,0.932659
7,0.089520,0.233954,0.933333,0.932659
8,0.068574,0.229475,0.933333,0.932659


TrainOutput(global_step=120, training_loss=0.5828936040401459, metrics={'train_runtime': 23.0947, 'train_samples_per_second': 83.136, 'train_steps_per_second': 5.196, 'total_flos': 63148354191360.0, 'train_loss': 0.5828936040401459, 'epoch': 8.0})

### 7. Model Fine-Tuning
Fine-tunes a multi-lingual BERT model on the synthetic dataset for automated ticket categorization.

real test

In [9]:
# ── Block 8: Test Fine-Tuned Category Model on Original Samples ─────────
category_model.eval()
dev = "cuda" if device == 0 else "cpu"

def predict_category(text):
    inputs = mbert_tokenizer(text, truncation=True, padding=True, max_length=64, return_tensors="pt").to(dev)
    with torch.no_grad():
        logits = category_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = torch.argmax(probs).item()
    return label_names[pred_idx], probs[pred_idx].item()

for i, c in enumerate(sample_complaints):
    cat, conf = predict_category(c)
    print(f"[{i}] {c}\n     PREDICTED: {cat} (confidence: {conf:.2f})\n")

[0] I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!
     PREDICTED: Billing (confidence: 0.94)

[1] الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت
     PREDICTED: Billing (confidence: 0.95)

[2] My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.
     PREDICTED: Network (confidence: 0.84)

[3] خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة
     PREDICTED: Technical Support (confidence: 0.30)

[4] The new mobile plan is a total ripoff, I'm paying more for less data than last month.
     PREDICTED: Sales/Plans (confidence: 0.93)



### 8. Summarization Logic
Sets up both abstractive (mT5/BART) and extractive summarizers, adding fallbacks for short or punctuation-less text.

sys1

In [10]:
# ── Block 9b: Sys 1 Summarizer (mT5 XLSum — multilingual) ────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sum_tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/mT5_multilingual_XLSum")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("csebuetnlp/mT5_multilingual_XLSum").to(
    "cuda" if device == 0 else "cpu"
).eval()

def sys1_summarize(text, max_length=60, min_length=10):
    dev = "cuda" if device == 0 else "cpu"
    inputs = sum_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(dev)
    with torch.no_grad():
        output_ids = sum_model.generate(
            **inputs, max_length=max_length, min_length=min_length,
            num_beams=4, no_repeat_ngram_size=2
        )
    return sum_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("sys1_summarize defined (mT5 XLSum).")

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

sys1_summarize defined (mT5 XLSum).


In [11]:
# ── Block 9 (Attempt 2): Try a Dialogue/Short-Text Summarizer ────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

longer_samples = [
    "I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating! I called customer service three times this week and every time they told me someone would call me back within 24 hours, but nobody ever did. I just want my money refunded as soon as possible.",
    "الشبكة عندي واقعة من خمس ايام ومفيش حد بيرد عليا. اتصلت بخدمة العملاء اكتر من مرة وكل مرة بيقولولي هيتحل خلال يوم واحد بس المشكلة لسه موجودة. انا محتاج الانترنت للشغل وده بيأثر عليا جدا، عايز حل سريع من فضلكم."
]

sum_tokenizer2 = AutoTokenizer.from_pretrained("philschmid/bart-large-cnn-samsum")
sum_model2 = AutoModelForSeq2SeqLM.from_pretrained("philschmid/bart-large-cnn-samsum").to(
    "cuda" if device == 0 else "cpu"
).eval()

def summarize_v2(text, max_length=60):
    dev = "cuda" if device == 0 else "cpu"
    inputs = sum_tokenizer2(text, return_tensors="pt", truncation=True, max_length=512).to(dev)
    with torch.no_grad():
        output_ids = sum_model2.generate(
            **inputs, max_length=max_length, min_length=0,
            num_beams=4, no_repeat_ngram_size=2,
            length_penalty=2.0
        )
    return sum_tokenizer2.decode(output_ids[0], skip_special_tokens=True)

print("ORIGINAL:", longer_samples[0])
print("SUMMARY:", summarize_v2(longer_samples[0]))

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

ORIGINAL: I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating! I called customer service three times this week and every time they told me someone would call me back within 24 hours, but nobody ever did. I just want my money refunded as soon as possible.
SUMMARY: I've been charged twice this month for my internet bill and no one is responding to my emails. I just want my money refunded as soon as possible.


In [12]:
def extractive_summarize(text, num_sentences=1):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        return text
    vec = TfidfVectorizer().fit_transform(sentences)
    sim_matrix = cosine_similarity(vec)
    scores = sim_matrix.sum(axis=1)
    scores[0] *= 1.5  # lead bias: first sentence often states the core issue
    top_idx = sorted(np.argsort(scores)[-num_sentences:])
    return " ".join([sentences[i] for i in top_idx])

In [13]:
# ── Step 3: Clause-level fallback splitting for punctuation-less Arabic ───

def split_clauses(text):
    # Fallback when there's no sentence-ending punctuation:
    # split on commas / Arabic comma / semicolons instead
    parts = re.split(r'[,،;]\s*', text.strip())
    return [p.strip() for p in parts if p.strip()]

def extractive_summarize(text, num_sentences=1):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        # No sentence boundaries found — try clause-level splitting
        clauses = split_clauses(text)
        if len(clauses) <= num_sentences:
            return text  # genuinely nothing to split on, return as-is
        sentences = clauses

    vec = TfidfVectorizer().fit_transform(sentences)
    sim_matrix = cosine_similarity(vec)
    scores = sim_matrix.sum(axis=1)
    scores[0] *= 1.5
    top_idx = sorted(np.argsort(scores)[-num_sentences:])
    return " ".join([sentences[i] for i in top_idx])

print("extractive_summarize updated: falls back to clause-level split when no sentence punctuation found.")

extractive_summarize updated: falls back to clause-level split when no sentence punctuation found.


In [14]:
# ── Step 4: Conjunction-word fallback + guaranteed shortening ─────────────

def split_conjunctions(text):
    # Last-resort split on standalone Arabic discourse markers (no punctuation needed)
    parts = re.split(r'\s+(?:بس|لكن|لاكن)\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def extractive_summarize(text, num_sentences=1):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        clauses = split_clauses(text)
        if len(clauses) > num_sentences:
            sentences = clauses
        else:
            conj = split_conjunctions(text)
            if len(conj) > num_sentences:
                sentences = conj
            else:
                # Truly nothing to split on — truncate proportionally instead
                # of returning the full text unchanged
                words = text.split()
                cut = max(8, int(len(words) * 0.6))
                return " ".join(words[:cut]) + ("…" if cut < len(words) else "")

    vec = TfidfVectorizer().fit_transform(sentences)
    sim_matrix = cosine_similarity(vec)
    scores = sim_matrix.sum(axis=1)
    scores[0] *= 1.15
    top_idx = sorted(np.argsort(scores)[-num_sentences:])
    return " ".join([sentences[i] for i in top_idx])

print("extractive_summarize updated: added conjunction-word fallback + proportional truncation as last resort.")

extractive_summarize updated: added conjunction-word fallback + proportional truncation as last resort.


In [15]:
# ── Step 1: Language-Routed Sys 1 Summarizer (fixes Arabic hallucination) ──

def sys1_summarize(text, max_length=60, min_length=8):
    lang = "ar" if is_arabic(text) else "en"
    if lang == "en":
        return summarize_v2(text, max_length=max_length, min_length=min_length)
    else:
        return extractive_summarize(text, num_sentences=1)

print("sys1_summarize updated: English -> BART, Arabic -> extractive.")

sys1_summarize updated: English -> BART, Arabic -> extractive.


In [16]:
# ── Step 2 (fix): Length-aware gating, word-count only ────────────────────

def sys1_summarize(text, max_length=None, min_length=None):
    lang = "ar" if is_arabic(text) else "en"
    n_words = len(text.split())
    if n_words < 8:
        return text
    if lang == "en":
        cap = max_length or (n_words + 15)
        return summarize_v2(text, max_length=cap)
    else:
        n_sent = len(split_sentences(text))
        k = 1 if n_sent <= 2 else 2
        return extractive_summarize(text, num_sentences=k)

print("sys1_summarize updated: dynamic num_sentences based on real sentence count.")


sys1_summarize updated: dynamic num_sentences based on real sentence count.


### 9. System 1 Pipeline
Combines sentiment, classification, and summarization into a single callable function.

In [17]:
# ── Block 10: Sys 1 — Combined Pipeline ──────────────────────────────────
def sys1_pipeline(text):
    # Sentiment
    sent_raw = sys1_sentiment(text[:512])[0]
    stars = int(sent_raw["label"][0])
    sentiment = "Positive" if stars >= 4 else "Neutral" if stars == 3 else "Negative"

    # Category
    category, cat_conf = predict_category(text)

    # Summary (only if text is reasonably long, else just return as-is)
    if len(text.split()) > 15:
        summary = sys1_summarize(text)
    else:
        summary = text

    return {
        "text": text,
        "sentiment": sentiment,
        "sentiment_raw": sent_raw,
        "category": category,
        "category_confidence": round(cat_conf, 2),
        "summary": summary
    }

# Run on all samples
import json
for c in sample_complaints:
    result = sys1_pipeline(c)
    print(json.dumps(result, ensure_ascii=False, indent=2), "\n")

{
  "text": "I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!",
  "sentiment": "Negative",
  "sentiment_raw": {
    "label": "1 star",
    "score": 0.9032513499259949
  },
  "category": "Billing",
  "category_confidence": 0.94,
  "summary": "I've been charged twice this month for my internet bill and no one is responding to my emails."
} 

{
  "text": "الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت",
  "sentiment": "Negative",
  "sentiment_raw": {
    "label": "1 star",
    "score": 0.49976158142089844
  },
  "category": "Billing",
  "category_confidence": 0.95,
  "summary": "الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت"
} 

{
  "text": "My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.",
  "sentiment": "Negative",
  "sentiment_raw": {
    "label": "1 star",


### 10. System 2: Specialized Models
Implements language-specific sentiment models (AraBERT and DistilBERT) for higher accuracy in mixed-language contexts.

In [18]:
# ── Block 11: Sys 2 — Load Separate EN and AR Models ─────────────────────

# English sentiment (dedicated English BERT/DistilBERT fine-tuned on sentiment)
sys2_sentiment_en = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

# Arabic sentiment (AraBERT fine-tuned on Arabic sentiment)
sys2_sentiment_ar = pipeline(
    "sentiment-analysis",
    model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
    device=device
)

# Quick isolated test
print("EN:", sys2_sentiment_en("The new mobile plan is a total ripoff."))
print("AR:", sys2_sentiment_ar("خدمة ممتازة جدا والفريق كان متعاون جدا"))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

EN: [{'label': 'NEGATIVE', 'score': 0.9996765851974487}]
AR: [{'label': 'positive', 'score': 0.9940899014472961}]


In [19]:
# ── Block 12: Sys 2 — Language-Routed Sentiment Aggregation ──────────────
def normalize_sentiment(label):
    label = label.upper()
    if label in ("POSITIVE",) or label == "5 STARS":
        return "Positive"
    if label in ("NEGATIVE",):
        return "Negative"
    return "Neutral"

def sys2_sentiment(text):
    chunks = detect_lang_chunks(text)  # from Block 3
    chunk_results = []
    for lang, chunk in chunks:
        if lang == "ar":
            r = sys2_sentiment_ar(chunk)[0]
        else:
            r = sys2_sentiment_en(chunk)[0]
        chunk_results.append({
            "lang": lang,
            "chunk": chunk,
            "label": normalize_sentiment(r["label"]),
            "score": round(r["score"], 3)
        })

    # Aggregate: majority vote weighted by confidence
    scores = {"Positive": 0.0, "Negative": 0.0, "Neutral": 0.0}
    for r in chunk_results:
        scores[r["label"]] += r["score"]
    overall = max(scores, key=scores.get)

    return {"overall_sentiment": overall, "chunks": chunk_results}

for i, c in enumerate(sample_complaints):
    result = sys2_sentiment(c)
    print(f"[{i}] {c}")
    print(f"     OVERALL: {result['overall_sentiment']}")
    for r in result["chunks"]:
        print(f"       - ({r['lang']}) \"{r['chunk']}\" → {r['label']} ({r['score']})")
    print()

[0] I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!
     OVERALL: Negative
       - (en) "I've been charged twice this month for my internet bill and no one is responding to my emails." → Negative (0.999)
       - (en) "This is so frustrating!" → Negative (0.999)

[1] الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت
     OVERALL: Negative
       - (ar) "الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت" → Negative (0.967)

[2] My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.
     OVERALL: Negative
       - (ar) "My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل." → Neutral (0.609)
       - (en) "I need this fixed today." → Negative (0.999)

[3] خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة
     

In [20]:
# ── Block 13: Sys 2 — Split Dataset by Language & Fine-Tune Separate Models ─

# Split existing df by whether text is majority-Arabic or majority-English
def is_arabic(text):
    arabic_chars = sum(1 for ch in text if '\u0600' <= ch <= '\u06FF')
    return arabic_chars > len(text) * 0.3

df["lang"] = df["text"].apply(lambda t: "ar" if is_arabic(t) else "en")
print(df["lang"].value_counts())

df_en = df[df["lang"] == "en"].reset_index(drop=True)
df_ar = df[df["lang"] == "ar"].reset_index(drop=True)

train_en, test_en = train_test_split(df_en, test_size=0.2, random_state=42, stratify=df_en["label"])
train_ar, test_ar = train_test_split(df_ar, test_size=0.2, random_state=42, stratify=df_ar["label"])

print(f"EN train/test: {len(train_en)}/{len(test_en)}")
print(f"AR train/test: {len(train_ar)}/{len(test_ar)}")

# Tokenizers for each language-specific BERT
en_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
ar_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")

train_dataset_en = ComplaintDataset(train_en["text"], train_en["label"], en_tokenizer)
test_dataset_en = ComplaintDataset(test_en["text"], test_en["label"], en_tokenizer)
train_dataset_ar = ComplaintDataset(train_ar["text"], train_ar["label"], ar_tokenizer)
test_dataset_ar = ComplaintDataset(test_ar["text"], test_ar["label"], ar_tokenizer)

print("Datasets ready.")

lang
en    154
ar    146
Name: count, dtype: int64
EN train/test: 123/31
AR train/test: 116/30


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/720k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Datasets ready.


In [21]:
# ── Block 14: Sys 2 — Fine-Tune EN and AR Category Classifiers ───────────

def train_category_model(model_name, train_ds, test_ds, num_labels, out_dir):
    model = BertForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    ).to("cuda" if device == 0 else "cpu")

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        learning_rate=2e-5,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics
    )
    trainer.train()
    return model

print("=== Training EN Category Model ===")
category_model_en = train_category_model(
    "bert-base-uncased", train_dataset_en, test_dataset_en, num_labels, "./cat_en_ckpt"
)

print("\n=== Training AR Category Model ===")
category_model_ar = train_category_model(
    "aubmindlab/bert-base-arabertv2", train_dataset_ar, test_dataset_ar, num_labels, "./cat_ar_ckpt"
)

=== Training EN Category Model ===


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.382845,0.580645,0.515205
2,1.521349,1.175871,0.870968,0.881989
3,1.259267,1.044036,0.870968,0.864473
4,1.047253,0.898209,0.967742,0.968485
5,0.886396,0.768072,1.000000,1.000000
6,0.886396,0.677373,1.000000,1.000000
7,0.751576,0.602994,1.000000,1.000000
8,0.671011,0.543278,1.000000,1.000000
9,0.562707,0.504777,1.000000,1.000000
10,0.561110,0.492163,1.000000,1.000000



=== Training AR Category Model ===


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.289811,0.800000,0.796104
2,1.509115,0.983954,0.966667,0.963636
3,1.173370,0.777059,0.966667,0.969231
4,0.860010,0.626171,0.966667,0.969231
5,0.602872,0.489206,0.966667,0.969231
6,0.602872,0.394837,0.966667,0.969231
7,0.490128,0.338414,0.966667,0.969231
8,0.367963,0.296357,0.966667,0.969231
9,0.316499,0.273272,0.966667,0.969231
10,0.296076,0.265444,0.966667,0.969231


In [22]:
# ── Block 15: Sys 2 — Test Category Models on Original Samples ───────────
category_model_en.eval()
category_model_ar.eval()
dev = "cuda" if device == 0 else "cpu"

def predict_category_sys2(text):
    lang = "ar" if is_arabic(text) else "en"
    if lang == "ar":
        inputs = ar_tokenizer(text, truncation=True, padding=True, max_length=64, return_tensors="pt").to(dev)
        model = category_model_ar
    else:
        inputs = en_tokenizer(text, truncation=True, padding=True, max_length=64, return_tensors="pt").to(dev)
        model = category_model_en
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = torch.argmax(probs).item()
    return lang, label_names[pred_idx], probs[pred_idx].item()

for i, c in enumerate(sample_complaints):
    lang, cat, conf = predict_category_sys2(c)
    print(f"[{i}] ({lang}) {c}\n     PREDICTED: {cat} (confidence: {conf:.2f})\n")

[0] (en) I've been charged twice this month for my internet bill and no one is responding to my emails. This is so frustrating!
     PREDICTED: Network (confidence: 0.36)

[1] (ar) الفاتورة بتاعتي غلط للمرة التالتة، دفعت المبلغ واتخصم مرتين وحد يرد عليا في اسرع وقت
     PREDICTED: Billing (confidence: 0.73)

[2] (ar) My internet has been down for 3 days now, وكل ما اتصل بالخدمة بيقولولي هيتحل خلال 24 ساعة ومفيش حاجة بتحصل. I need this fixed today.
     PREDICTED: Network (confidence: 0.35)

[3] (ar) خدمة ممتازة جدا والفريق كان متعاون جدا في حل المشكلة بسرعة
     PREDICTED: Other (confidence: 0.35)

[4] (en) The new mobile plan is a total ripoff, I'm paying more for less data than last month.
     PREDICTED: Sales/Plans (confidence: 0.56)



### 11. System 2 Summarizer
Uses dedicated models for each language: `bart-large-cnn` for English and an embedding-based extractive approach for Arabic.

In [23]:
# ── Block 18: Sys 2 — Dedicated Summarizers (EN: BART-CNN, AR: AraBERT-extractive) ─
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel

dev = "cuda" if device == 0 else "cpu"

sys2_sum_tokenizer_en = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sys2_sum_model_en = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(dev).eval()

arabert_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
arabert_model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv2").to(dev).eval()

def arabert_extractive_summarize(text, num_sentences=1):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        clauses = split_clauses(text)
        if len(clauses) > num_sentences:
            sentences = clauses
        else:
            conj = split_conjunctions(text)
            if len(conj) > num_sentences:
                sentences = conj
            else:
                words = text.split()
                cut = max(8, int(len(words) * 0.6))
                return " ".join(words[:cut]) + ("…" if cut < len(words) else "")

    inputs = arabert_tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=64).to(dev)
    with torch.no_grad():
        outputs = arabert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    sim_matrix = cosine_similarity(embeddings)
    scores = sim_matrix.sum(axis=1)
    scores[0] *= 1.15
    top_idx = sorted(np.argsort(scores)[-num_sentences:])
    return " ".join([sentences[i] for i in top_idx])

def sys2_summarize(text, max_length=None):
    n_words = len(text.split())
    if n_words < 8:
        return text

    lang = "ar" if is_arabic(text) else "en"
    if lang == "ar":
        n_sent = len(split_sentences(text))
        k = 1 if n_sent <= 2 else 2
        return arabert_extractive_summarize(text, num_sentences=k)

    cap = max_length or (n_words + 15)
    inputs = sys2_sum_tokenizer_en(text, return_tensors="pt", truncation=True, max_length=512).to(dev)
    with torch.no_grad():
        output_ids = sys2_sum_model_en.generate(
            **inputs, max_length=cap, min_length=0,
            num_beams=4, no_repeat_ngram_size=2, length_penalty=1.0
        )
    return sys2_sum_tokenizer_en.decode(output_ids[0], skip_special_tokens=True)

print("sys2_summarize: EN = bart-large-cnn, AR = AraBERT-embedding extractive (no hallucination possible).")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sys2_summarize: EN = bart-large-cnn, AR = AraBERT-embedding extractive (no hallucination possible).


In [24]:
# ── Block 18.1: Sys 2.1 — Alt English Summarizer (EN: BART-CNN-SAMSum, AR: AraBERT-extractive) ─
def sys2_1_summarize(text, max_length=None):
    n_words = len(text.split())
    if n_words < 8:
        return text

    lang = "ar" if is_arabic(text) else "en"
    if lang == "ar":
        n_sent = len(split_sentences(text))
        k = 1 if n_sent <= 2 else 2
        return arabert_extractive_summarize(text, num_sentences=k)

    cap = max_length or (n_words + 15)
    return summarize_v2(text, max_length=cap)

print("sys2_1_summarize: EN = bart-large-cnn-samsum, AR = AraBERT-embedding extractive.")

sys2_1_summarize: EN = bart-large-cnn-samsum, AR = AraBERT-embedding extractive.


In [25]:
# ── Block 19: Sys 2 — Combined Pipeline ────────────────────────────────────
def sys2_pipeline(text):
    sent_result = sys2_sentiment(text)
    sentiment = sent_result["overall_sentiment"]

    _, category, cat_conf = predict_category_sys2(text)
    summary = sys2_summarize(text)

    return {
        "text": text,
        "sentiment": sentiment,
        "sentiment_detail": sent_result["chunks"],  # kept for inspection, not printed by Block 17
        "category": category,
        "category_confidence": round(cat_conf, 2),
        "summary": summary
    }

print("sys2_pipeline defined.")

sys2_pipeline defined.


In [26]:
# ── Block 19.1: Sys 2.1 — Combined Pipeline ────────────────────────────────
def sys2_1_pipeline(text):
    sent_result = sys2_sentiment(text)
    sentiment = sent_result["overall_sentiment"]

    _, category, cat_conf = predict_category_sys2(text)
    summary = sys2_1_summarize(text)

    return {
        "text": text,
        "sentiment": sentiment,
        "sentiment_detail": sent_result["chunks"],
        "category": category,
        "category_confidence": round(cat_conf, 2),
        "summary": summary
    }

print("sys2_1_pipeline defined.")


sys2_1_pipeline defined.


### 12. Evaluation & Comparison
Runs the test set through all system versions and calculates agreement rates and compression metrics.

In [27]:
# ── Block 17: Evaluation Test Set — Run All Systems ──────────────────────
test_set = [
    "I've been charged twice for my internet bill this month and support isn't responding. Please refund me immediately.",
    "الانترنت عندي واقع من اربع ايام والفني اللي جه ماصلحش المشكلة، محتاج حد يحل الموضوع بسرعة",
    "My router stopped working after the last update and I've tried resetting it three times with no luck.",
    "الباقة الجديدة غالية جدا وفيها بيانات اقل من اللي كنت مشترك فيها قبل كده، مش عادل خالص",
    "Just wanted to thank the support agent, she was very patient and fixed my issue in minutes.",
    "عايز اقفل حسابي نهائيا بس مش لاقي الخيار ده في التطبيق، ممكن تساعدوني؟",
    "The signal in my area drops constantly, especially in the evenings, and it's been like this for two weeks now.",
    "دفعت الفاتورة بس النظام لسه بيقول اني متأخر في السداد وده مش صح"
]

for i, text in enumerate(test_set):
    r1 = sys1_pipeline(text)
    r2 = sys2_pipeline(text)
    r2_1 = sys2_1_pipeline(text)
    print(f"[{i}] {text}")
    print(f"   SYS1   → sentiment: {r1['sentiment']}, category: {r1['category']} ({r1['category_confidence']}), summary: {r1['summary']}")
    print(f"   SYS2   → sentiment: {r2['sentiment']}, category: {r2['category']} ({r2['category_confidence']}), summary: {r2['summary']}")
    print(f"   SYS2.1 → sentiment: {r2_1['sentiment']}, category: {r2_1['category']} ({r2_1['category_confidence']}), summary: {r2_1['summary']}\n")


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[0] I've been charged twice for my internet bill this month and support isn't responding. Please refund me immediately.
   SYS1   → sentiment: Negative, category: Billing (0.94), summary: I've been charged twice for my internet bill this month and support isn't responding. Please refund me immediately.
   SYS2   → sentiment: Negative, category: Billing (0.37), summary: I've been charged twice for my internet bill this month and support isn't responding. Please refund me immediately.
   SYS2.1 → sentiment: Negative, category: Billing (0.37), summary: I've been charged twice for my internet bill this month and support isn't responding. Please refund me immediately.

[1] الانترنت عندي واقع من اربع ايام والفني اللي جه ماصلحش المشكلة، محتاج حد يحل الموضوع بسرعة
   SYS1   → sentiment: Negative, category: Network (0.92), summary: الانترنت عندي واقع من اربع ايام والفني اللي جه ماصلحش المشكلة
   SYS2   → sentiment: Neutral, category: Network (0.3), summary: الانترنت عندي واقع من اربع ايام والفن

In [28]:
# ── Block 20: Combined Eval Set ────────────────────────────────────────────
eval_set = sample_complaints + test_set

eval_results = []
for text in eval_set:
    r1 = sys1_pipeline(text)
    r2 = sys2_pipeline(text)
    r2_1 = sys2_1_pipeline(text)
    eval_results.append({"text": text, "sys1": r1, "sys2": r2, "sys2_1": r2_1})

print(f"Ran {len(eval_results)} complaints through all 3 systems.")

Ran 13 complaints through all 3 systems.


In [29]:
# ── Block 21: Agreement & Summary-Length Metrics ───────────────────────────
def agreement_rate(results, key, a, b):
    matches = sum(1 for r in results if r[a][key] == r[b][key])
    return round(matches / len(results), 2)

def avg_compression(results, sys_key):
    ratios = []
    for r in results:
        orig_words = len(r["text"].split())
        sum_words = len(r[sys_key]["summary"].split())
        if orig_words > 0:
            ratios.append(sum_words / orig_words)
    return round(sum(ratios) / len(ratios), 2)

print("=== Sentiment Agreement ===")
print("SYS1 vs SYS2:  ", agreement_rate(eval_results, "sentiment", "sys1", "sys2"))
print("SYS1 vs SYS2.1:", agreement_rate(eval_results, "sentiment", "sys1", "sys2_1"))
print("SYS2 vs SYS2.1:", agreement_rate(eval_results, "sentiment", "sys2", "sys2_1"))

print("\n=== Category Agreement ===")
print("SYS1 vs SYS2:  ", agreement_rate(eval_results, "category", "sys1", "sys2"))
print("SYS1 vs SYS2.1:", agreement_rate(eval_results, "category", "sys1", "sys2_1"))
print("SYS2 vs SYS2.1:", agreement_rate(eval_results, "category", "sys2", "sys2_1"))

print("\n=== Avg Summary Length (as fraction of original) ===")
print("SYS1:  ", avg_compression(eval_results, "sys1"))
print("SYS2:  ", avg_compression(eval_results, "sys2"))
print("SYS2.1:", avg_compression(eval_results, "sys2_1"))

=== Sentiment Agreement ===
SYS1 vs SYS2:   0.69
SYS1 vs SYS2.1: 0.69
SYS2 vs SYS2.1: 1.0

=== Category Agreement ===
SYS1 vs SYS2:   0.77
SYS1 vs SYS2.1: 0.77
SYS2 vs SYS2.1: 1.0

=== Avg Summary Length (as fraction of original) ===
SYS1:   0.84
SYS2:   0.79
SYS2.1: 0.69


### 13. Interactive UI
Launches a Gradio interface to allow users to input text or upload files and compare the performance of each NLP pipeline.

In [30]:
# ── Block 22: Simple Gradio UI ─────────────────────────────────────────────
!pip install -q gradio
import gradio as gr

def run_all_systems(text_input, file_input):
    text = text_input.strip() if text_input else ""
    if file_input is not None:
        with open(file_input.name, "r", encoding="utf-8") as f:
            text = f.read().strip()

    if not text:
        return "Please enter text or upload a .txt file.", "", ""

    r1 = sys1_pipeline(text)
    r2 = sys2_pipeline(text)
    r2_1 = sys2_1_pipeline(text)

    def fmt(r, name):
        return f"**{name}**\nSentiment: {r['sentiment']}\nCategory: {r['category']} ({r['category_confidence']})\nSummary: {r['summary']}"

    return fmt(r1, "SYS1"), fmt(r2, "SYS2"), fmt(r2_1, "SYS2.1")

demo = gr.Interface(
    fn=run_all_systems,
    inputs=[
        gr.Textbox(label="Paste complaint text", lines=4),
        gr.File(label="...or upload a .txt file", file_types=[".txt"])
    ],
    outputs=[
        gr.Markdown(label="SYS1 Output"),
        gr.Markdown(label="SYS2 Output"),
        gr.Markdown(label="SYS2.1 Output")
    ],
    title="Complaint Classification, Summarization & Sentiment — Sys1 vs Sys2 vs Sys2.1",
    description="Enter a complaint (English, Arabic, or mixed) or upload a .txt file to compare all three systems."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://93e9d875d06a0fb92d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
